# III. Binary classification and optimization

## Which model predicts a repeat purchase within 90 days?

Each row is a purchase. At the time of that purchase, we predict whether the same customer will buy again within 90 days. We compare eight classifiers on the same purchases, first with transaction information and then with four additional customer attributes.

[Data](#data) · [Time windows](#windows) · [Models and metric](#models) · [Starting results](#results) · [Tuning](#tuning) · [Forest errors and purchase rhythm](#forest-review) · [Four additional relationships](#new-relations) · [Forest feature groups](#groups) · [IV. Frequency and errors](#segments) · [Threshold and training history](#threshold-history) · [V. Deployment](#kaggle-rf)

<a id="data"></a>

## 4.1 Preparing the purchase data

Notebook 01 identified 65 rows with neither a date nor transaction predictors. We exclude those rows and apply its historical corrections to the remaining purchases. The shared cleaning function uses counts and amounts from the same row to recover exact values. It leaves unresolved inconsistencies missing. We keep the purchase ID for submission, the customer ID for joining tables and the date for splitting by time.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier

root = Path.cwd()
if not (root / 'data').is_dir():
    root = root.parent
sys.path.insert(0, str(root))
from src.history_cleaning import recover_time_intervals, repair_history
from src.final_forest import (CLIENT_FEATURES, SELECTED_FILE, add_client_features, client_forest,
                              export_forest, make_preparation, transaction_columns, write_selected_submission)

target = 'repeat_purchase_90d'
raw = pd.read_csv(root / 'data' / 'train.csv')
clients = pd.read_csv(root / 'data' / 'clientes.csv')
dates = pd.to_datetime(raw['purchase_date'], errors='raise')
predictors = raw.columns.drop(['ID', 'customer_id', target, 'purchase_date'])
empty_rows = raw[predictors].isna().all(axis=1)
assert empty_rows.equals(dates.isna())
assert raw['ID'].is_unique and raw[target].isin([0, 1]).all()

data, audit = repair_history(raw.loc[dates.notna()].copy())
data['purchase_date'] = dates.loc[data.index]
data = data.reset_index(drop=True)
print('Eligible purchases:', len(data))
print('Purchases with inconsistent historical values:', audit['Distinct rows with invalid values'])

Eligible purchases: 6469
Purchases with inconsistent historical values: 227


### Recovering missing time intervals

This additional rule assumes that tenure starts at the first purchase and that `prior_mean_gap_days` averages the gaps **between previous purchases**. Three previous purchases give two such gaps. In one record, tenure is 133 days and the two gaps average 14.5 days. The last previous purchase was therefore 133 − 2 × 14.5 = 104 days ago.

If either recency or the mean gap is known, we can recover the other when at least two prior purchases exist. With one prior purchase, recency equals tenure and the mean gap is undefined. The code checks the identity wherever both quantities are known. Rounding recency removes floating-point error because purchase dates have day precision.

In [2]:
time_columns = ['days_since_previous_purchase', 'prior_mean_gap_days']
original_times = data[time_columns].copy()
data[time_columns] = recover_time_intervals(data)
recovered = original_times.isna().sum() - data[time_columns].isna().sum()
display(recovered.rename('Recovered values').to_frame())

,Recovered values
days_since_previous_purchase,201
prior_mean_gap_days,203


In [3]:
client_features = CLIENT_FEATURES
purchases = add_client_features(data, clients)
transaction_features = transaction_columns(data)
feature_sets = {
    'Transactions': transaction_features,
    'Transactions + clients': transaction_features + client_features,
}
y = purchases[target]
print('Transaction inputs:', len(transaction_features))
print('Extra customer inputs:', len(client_features))

Transaction inputs: 34
Extra customer inputs: 4


The join must keep exactly one row per purchase in its original order. Entry year is a category so that each year can have its own effect. Using these four attributes assumes that their recorded values were available at purchase time. The client file has no dated versions to verify changes in region or island.

Notebook 03 tested `random_noise` and its final logistic selector discarded it. We omit it from both input sets. We keep the other transaction variables so that a feature discarded by logistic regression can still contribute to a tree split or a neural network. The same purchases enter both comparisons.

<a id="windows"></a>

## 4.2 Training on the past and validating on later purchases

We keep the main split from Notebook 03. The purchase at the 70% position in date order falls on 13 July 2022. A purchase contributes to training only if its entire 90-day outcome was observable before the validation period began. This leaves 4,424 purchases for model development, 103 in the gap and 1,942 in the later block. Notebook 03 already evaluated logistic regression on that later block.

Inside the 4,424 purchases, we move the prediction date forward three times. The first two validation windows cover a year each. The third ends at the original training cutoff. These dates give three consecutive periods of comparable size. For every model we use the same boundaries and a gap of 90 **calendar days**. Earlier validation purchases can enter a later training set once their outcomes are known. Customers can appear again because the task includes returning customers.

In [4]:
date = purchases['purchase_date']
horizon = pd.Timedelta(days=90)
main_start = date.sort_values().iloc[int(len(date) * 0.70)]
development_end = main_start - horizon
boundaries = [pd.Timestamp('2019-07-01'), pd.Timestamp('2020-07-01'),
              pd.Timestamp('2021-07-01'), development_end]

folds = []
periods = []
for start, end in zip(boundaries, boundaries[1:]):
    train_rows = date < start - horizon
    gap_rows = (date >= start - horizon) & (date < start)
    valid_rows = (date >= start) & (date < end)
    assert date[train_rows].max() + horizon < date[valid_rows].min()
    assert date[valid_rows].max() < development_end
    assert y[train_rows].nunique() == 2 and y[valid_rows].nunique() == 2
    folds.append((train_rows, valid_rows))
    periods.append([date[train_rows].max().date(), start.date(), (end - pd.Timedelta(days=1)).date(),
                    train_rows.sum(), gap_rows.sum(), valid_rows.sum(), y[valid_rows].mean()])
display(pd.DataFrame(periods, columns=['Last training date', 'Validation from', 'Through', 'Train', 'Gap',
                                      'Validation', 'Repeat rate']).round(3))

,Last training date,Validation from,Through,Train,Gap,Validation,Repeat rate
0,2019-04-01,2019-07-01,2020-06-30,2287,86,707,0.546
1,2020-03-31,2020-07-01,2021-06-30,2962,118,692,0.595
2,2021-03-30,2021-07-01,2022-04-13,3649,123,652,0.514


<a id="models"></a>

## 4.3 Models, preprocessing and score

In each training window we learn numerical medians, missing-value indicators and scaling. For categories we fill missing entries with `Missing` and learn one-hot columns. An unseen category can still be transformed. Scaling matters for KNN distances and neural network optimisation. We keep the encoded table dense so Gaussian Naive Bayes can accept it too. Scaling also lets us use the same preparation across all families.

We measure **F1 for repeat purchase**, the competition metric. A missed repeat purchase and a needless return prediction both matter when planning follow-up. We compute it separately for each period and average the three scores equally, so one larger period cannot dominate. Predicting a return for every purchase is our reference.

We apply thresholds of 0.50 and 0.45 to the same probabilities. At 0.45, an extra purchase counts as positive when its probability falls between these two cutoffs. Whether that helps F1 depends on how many of those extra positives actually returned.

In [5]:
preparation = make_preparation()

thresholds = [0.50, 0.45]
window_names = ['Period 1', 'Period 2', 'Period 3']

def window_probabilities(classifier, inputs, columns_by_period=None, windows=None):
    if windows is None:
        windows = folds
    probabilities = []
    if columns_by_period is None:
        columns_by_period = [inputs.columns for _ in windows]
    for (train_rows, valid_rows), columns in zip(windows, columns_by_period, strict=True):
        model = make_pipeline(clone(preparation), clone(classifier))
        model.fit(inputs.loc[train_rows, columns], y.loc[train_rows])
        valid_inputs = inputs.loc[valid_rows, columns]
        values = model.predict_proba(valid_inputs)[:, 1]
        probabilities.append(pd.Series(values, index=valid_inputs.index))
    return probabilities

def period_scores(classifier, inputs):
    probabilities = window_probabilities(classifier, inputs)
    return {threshold: [f1_score(y.loc[p.index], p >= threshold) for p in probabilities]
            for threshold in thresholds}

### What each model adds to the comparison

Logistic regression provides a linear reference. Gaussian Naive Bayes models variables separately within each class. KNN predicts from nearby purchases. A decision tree learns conditional splits. A random forest averages trees, while gradient boosting builds trees in sequence to correct earlier predictions.

In Week 11, histogram gradient boosting groups numeric values into bins before choosing tree splits. We start with 100 boosting iterations and up to 31 leaves per tree. In Weeks 12 and 13, the MLP learns a nonlinear relationship across inputs. We use one hidden layer with 16 units, ReLU activation and the Adam optimiser. This keeps the network small for training windows of about 2,300 to 3,650 purchases. Its 1,000 iterations are an upper limit. Both new models use a fixed seed and disable validation-based early stopping so the calendar windows control evaluation. One seed makes the run reproducible but cannot measure seed-to-seed variation.

In [6]:
models = {
    'Logistic regression': LogisticRegression(max_iter=10000),
    'Gaussian Naive Bayes': GaussianNB(),
    'K nearest neighbours': KNeighborsClassifier(n_neighbors=5),
    'Decision tree': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.10, random_state=42),
    'Histogram gradient boosting': HistGradientBoostingClassifier(
        max_iter=100, max_leaf_nodes=31, early_stopping=False, random_state=42),
    'Small MLP': MLPClassifier(hidden_layer_sizes=(16,), activation='relu', solver='adam',
                               alpha=0.0001, max_iter=1000, early_stopping=False, random_state=42),
}

### Does the timing correction improve prediction?

First compare logistic regression with and without the two timing corrections. The folds, inputs and 0.50 threshold are fixed. This isolates the effect of changing those two columns, including their remaining missing values.

In [7]:
before = purchases[transaction_features].copy()
before[time_columns] = original_times
logistic = models['Logistic regression']
timing_check = pd.DataFrame([
    period_scores(logistic, before)[0.50],
    period_scores(logistic, purchases[transaction_features])[0.50],
], columns=window_names, index=['Notebook 01 rules', 'With timing recovery'])
timing_check['Mean F1'] = timing_check.mean(axis=1)
display(timing_check.round(4))

,Period 1,Period 2,Period 3,Mean F1
Notebook 01 rules,0.7233,0.7190,0.7108,0.7177
With timing recovery,0.7180,0.7206,0.7083,0.7157


Recovering the time intervals raised logistic F1 in period 2 and reduced it in periods 1 and 3. Mean F1 moved from 0.7177 to 0.7157. We retain the rule because it restores known historical values. This comparison gives no evidence of an average predictive gain for logistic regression.

<a id="results"></a>

## 4.4 Eight models with and without client data

Each starting configuration gets the same two input sets, three periods and two thresholds. We fit once per period and reuse the probabilities for both thresholds. The reference also varies by period because the proportion of positive outcomes changes.

In [8]:
reference = [f1_score(y.loc[valid], [1] * int(valid.sum())) for _, valid in folds]
always_positive = pd.Series(reference, index=window_names, name='Always positive F1')
always_positive['Mean F1'] = always_positive.mean()
display(always_positive.round(4).to_frame().T)

def compare_settings(settings):
    rows = []
    for name, changes in settings:
        classifier = clone(models[name]).set_params(**changes)
        setting = ', '.join(f'{key}={value}' for key, value in changes.items()) or 'Starting'
        for feature_set, columns in feature_sets.items():
            scores = period_scores(classifier, purchases[columns])
            for threshold in thresholds:
                rows.append([name, setting, feature_set, threshold, *scores[threshold]])
    table = pd.DataFrame(rows, columns=['Model', 'Setting', 'Inputs', 'Threshold', *window_names])
    table['Mean F1'] = table[window_names].mean(axis=1)
    return table

starting = compare_settings([(name, {}) for name in models])
paired = starting.pivot(index=['Model', 'Threshold'], columns='Inputs', values='Mean F1')
paired['Change with clients'] = paired['Transactions + clients'] - paired['Transactions']
display(paired.round(4))

,Period 1,Period 2,Period 3,Mean F1
Always positive F1,0.7063,0.7464,0.6788,0.7105


Inputs                                 Transactions  Transactions + clients  \
Model                       Threshold                                         
Decision tree               0.45             0.7075                  0.7101   
                            0.50             0.7077                  0.7053   
Gaussian Naive Bayes        0.45             0.7252                  0.7160   
                            0.50             0.7258                  0.7166   
Gradient boosting           0.45             0.7406                  0.7333   
                            0.50             0.7263                  0.7240   
Histogram gradient boosting 0.45             0.7164                  0.7113   
                            0.50             0.7149                  0.7087   
K nearest neighbours        0.45             0.6950                  0.7155   
                            0.50             0.6950                  0.7155   
Logistic regression         0.45             0.7323                  0.7254   
                            0.50             0.7157                  0.7033   
Random forest               0.45             0.7416                  0.7454   
                            0.50             0.7352                  0.7355   
Small MLP                   0.45             0.6503                  0.6626   
                            0.50             0.6372                  0.6535   

Inputs                                 Change with clients  
Model                       Threshold                       
Decision tree               0.45                    0.0026  
                            0.50                   -0.0024  
Gaussian Naive Bayes        0.45                   -0.0092  
                            0.50                   -0.0092  
Gradient boosting           0.45                   -0.0073  
                            0.50                   -0.0023  
Histogram gradient boosting 0.45                   -0.0051  
                            0.50                   -0.0063  
K nearest neighbours        0.45                    0.0205  
                            0.50                    0.0205  
Logistic regression         0.45                   -0.0069  
                            0.50                   -0.0123  
Random forest               0.45                    0.0039  
                            0.50                    0.0003  
Small MLP                   0.45                    0.0122  
                            0.50                    0.0163

The always-positive reference has mean F1 of 0.7105, ranging from 0.6788 to 0.7464 as the repeat rate changes. In the original six-model comparison at 0.50, random forest led and gradient boosting on transactions was the next family. These were the first two families we tuned.

At 0.45 with starting settings, histogram boosting reaches 0.7164 using transactions and 0.7113 with client attributes. The small MLP starts at 0.6503 and 0.6626 respectively. Keeping the input set fixed lets us check next whether a change in leaf size or regularisation helps either model.

<a id="tuning"></a>

## 4.5 Comparing a small set of settings

The first six-model benchmark led us to tune random forest and gradient boosting. We then added histogram gradient boosting and a small MLP. We keep the earlier scores as a reference and use the same protocol throughout.

| Family | Settings compared | Reason |
|---|---|---|
| Random forest | Minimum leaf size 1, 5 or 10 | Larger leaves average more purchases. |
| Gradient boosting | 100 or 200 trees and learning rate 0.05 or 0.10 | A smaller update can need more trees. |
| Histogram gradient boosting | At most 31 or 15 leaves per tree | Fewer leaves limit how specific a tree can be. |
| MLP | L2 penalty `alpha` of 0.0001, 0.01 or 1 | Stronger regularisation penalises large weights. The 16-unit architecture stays fixed. |

The table below combines the starting scores with the new fits. For each input set and setting, the two thresholds score exactly the same purchases. The gain column measures the effect of changing the threshold alone.

In [9]:
forest_name = 'Random forest'
boost_name = 'Gradient boosting'
hist_name = 'Histogram gradient boosting'
mlp_name = 'Small MLP'
changes_to_try = [
    (forest_name, {'min_samples_leaf': 5}),
    (forest_name, {'min_samples_leaf': 10}),
    (boost_name, {'learning_rate': 0.05, 'n_estimators': 100}),
    (boost_name, {'learning_rate': 0.05, 'n_estimators': 200}),
    (boost_name, {'learning_rate': 0.10, 'n_estimators': 200}),
    (hist_name, {'max_leaf_nodes': 15}),
    (mlp_name, {'alpha': 0.01}),
    (mlp_name, {'alpha': 1.0}),
]
extra = compare_settings(changes_to_try)
results = pd.concat([starting, extra], ignore_index=True)
tuned_families = [forest_name, boost_name, hist_name, mlp_name]
tuning = results[results['Model'].isin(tuned_families)]
by_threshold = tuning.pivot(index=['Model', 'Inputs', 'Setting'], columns='Threshold', values='Mean F1')
by_threshold['Gain at 0.45'] = by_threshold[0.45] - by_threshold[0.50]
display(by_threshold[[0.50, 0.45, 'Gain at 0.45']].round(4))

Threshold                                                                                   0.5  \
Model                       Inputs                 Setting                                        
Gradient boosting           Transactions           Starting                              0.7263   
                                                   learning_rate=0.05, n_estimators=100  0.7271   
                                                   learning_rate=0.05, n_estimators=200  0.7220   
                                                   learning_rate=0.1, n_estimators=200   0.7202   
                            Transactions + clients Starting                              0.7240   
                                                   learning_rate=0.05, n_estimators=100  0.7365   
                                                   learning_rate=0.05, n_estimators=200  0.7227   
                                                   learning_rate=0.1, n_estimators=200   0.7103   
Histogram gradient boosting Transactions           Starting                              0.7149   
                                                   max_leaf_nodes=15                     0.7170   
                            Transactions + clients Starting                              0.7087   
                                                   max_leaf_nodes=15                     0.7152   
Random forest               Transactions           Starting                              0.7352   
                                                   min_samples_leaf=10                   0.7459   
                                                   min_samples_leaf=5                    0.7352   
                            Transactions + clients Starting                              0.7355   
                                                   min_samples_leaf=10                   0.7442   
                                                   min_samples_leaf=5                    0.7442   
Small MLP                   Transactions           Starting                              0.6372   
                                                   alpha=0.01                            0.6362   
                                                   alpha=1.0                             0.7221   
                            Transactions + clients Starting                              0.6535   
                                                   alpha=0.01                            0.6548   
                                                   alpha=1.0                             0.7173   

Threshold                                                                                  0.45  \
Model                       Inputs                 Setting                                        
Gradient boosting           Transactions           Starting                              0.7406   
                                                   learning_rate=0.05, n_estimators=100  0.7377   
                                                   learning_rate=0.05, n_estimators=200  0.7397   
                                                   learning_rate=0.1, n_estimators=200   0.7243   
                            Transactions + clients Starting                              0.7333   
                                                   learning_rate=0.05, n_estimators=100  0.7368   
                                                   learning_rate=0.05, n_estimators=200  0.7360   
                                                   learning_rate=0.1, n_estimators=200   0.7249   
Histogram gradient boosting Transactions           Starting                              0.7164   
                                                   max_leaf_nodes=15                     0.7234   
                            Transactions + clients Starting                              0.7113   
                                                   max_leaf_nodes=15                     0.7241   
Random forest               Transaction

### Reading the best scores

Within a model and input set, choose the highest mean F1 at full precision. An exact tie favours threshold 0.50 and then the first setting tried. Each row below shows the winning setting for that model and input set, across all three periods. A small client advantage for two separately tuned winners can include the effects of both tuning and client data. The paired tables above isolate the effect of changing one choice at a time.

In [10]:
ranked = results.sort_values(['Mean F1', 'Threshold'], ascending=False, kind='stable')
best = ranked.drop_duplicates(['Model', 'Inputs'])
display(best.reset_index(drop=True).round(4))

,Model,Setting,Inputs,Threshold,Period 1,Period 2,Period 3,Mean F1
0,Random forest,min_samples_leaf=5,Transactions + clients,0.45,0.7521,0.7431,0.7516,0.7489
1,Random forest,min_samples_leaf=10,Transactions,0.45,0.7433,0.7468,0.7484,0.7462
2,Gradient boosting,Starting,Transactions,0.45,0.7242,0.7433,0.7543,0.7406
3,Gradient boosting,"learning_rate=0.05, n_estimators=100",Transactions + clients,0.45,0.7312,0.7473,0.7320,0.7368
4,Small MLP,alpha=1.0,Transactions,0.45,0.7439,0.7327,0.7304,0.7357
5,Logistic regression,Starting,Transactions,0.45,0.7356,0.7422,0.7190,0.7323
6,Small MLP,alpha=1.0,Transactions + clients,0.45,0.7112,0.7376,0.7330,0.7272
7,Gaussian Naive Bayes,Starting,Transactions,0.50,0.7245,0.7448,0.7081,0.7258
8,Logistic regression,Starting,Transactions + clients,0.45,0.7173,0.7431,0.7159,0.7254
9,Histogram gradient boosting,max_leaf_nodes=15,Transactions + clients,0.45,0.7137,0.7331,0.7255,0.7241


**Random forest remains first** with customer attributes, five purchases per leaf and threshold 0.45. Its F1 is 0.7521, 0.7431 and 0.7516 across the three periods (mean 0.7489). In period 2, predicting a return for every purchase scores 0.7464. The forest leads the reference on average, with a small loss in that period. For gradient boosting the original 100 trees and 0.10 learning rate on transactions give the best mean (0.7406) at 0.45.

**The MLP is sensitive to regularisation.** On transactions at the same 0.45 threshold, increasing `alpha` from 0.0001 to 1 raises mean F1 from 0.6503 to 0.7357. Adding clients with the same settings gives 0.7272. Histogram boosting performs best with at most 15 leaves and threshold 0.45. Its mean is 0.7234 on transactions and 0.7241 with clients. That small difference gives little basis for a general claim about client attributes.

Section 4.7 reproduces this forest submission and the purchase rhythm submission. The comparisons above use only the three internal periods.

<a id="forest-review"></a>

## 4.6 Where does the forest make mistakes?

We start with the exact settings previously submitted to Kaggle. Here we train fresh models on the three internal windows and count their validation errors. One positive prediction means that the customer is expected to buy again within 90 days of that purchase.

We report unnecessary follow-ups (false positives) and missed returns (false negatives). The pooled counts describe individual purchases. Model selection gives each of the three periods equal weight.

In [11]:
def error_breakdown(probability, threshold):
    actual = y.loc[probability.index]
    predicted = probability >= threshold
    tn, fp, fn, tp = confusion_matrix(actual, predicted, labels=[0, 1]).ravel()
    return {
        'Purchases': len(actual), 'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn,
        'Precision': precision_score(actual, predicted, zero_division=0),
        'Recall': recall_score(actual, predicted, zero_division=0),
        'F1': f1_score(actual, predicted, zero_division=0),
    }

client_inputs = feature_sets['Transactions + clients']
reference_forest = client_forest()
reference_probs = window_probabilities(reference_forest, purchases[client_inputs])
pooled_probs = pd.concat(reference_probs)
assert pooled_probs.index.is_unique
errors = [error_breakdown(p, 0.45) for p in reference_probs]
errors.append(error_breakdown(pooled_probs, 0.45))
display(pd.DataFrame(errors, index=window_names + ['Pooled purchases']).round(4))

prior = purchases.loc[pooled_probs.index, 'prior_purchase_count']
by_history = {
    'First purchase': error_breakdown(pooled_probs[prior == 0], 0.45),
    'Previous purchases': error_breakdown(pooled_probs[prior > 0], 0.45),
}
display(pd.DataFrame(by_history).T.round(4))

,Purchases,TP,FP,FN,TN,Precision,Recall,F1
Period 1,707,311,130,75,191,0.7052,0.8057,0.7521
Period 2,692,321,131,91,149,0.7102,0.7791,0.7431
Period 3,652,286,140,49,177,0.6714,0.8537,0.7516
Pooled purchases,2051,918,401,215,517,0.6960,0.8102,0.7488


,Purchases,TP,FP,FN,TN,Precision,Recall,F1
First purchase,134.0,9.0,22.0,26.0,77.0,0.2903,0.2571,0.2727
Previous purchases,1917.0,909.0,379.0,189.0,440.0,0.7057,0.8279,0.7619


Across 2,051 validation purchases, 918 predicted returns happened and 401 did not. The forest missed 215 returns and correctly predicted 517 negatives. Precision is 918 / 1,319 = 69.6% and recall is 918 / 1,133 = 81.0%. Together these give pooled F1 = 1,836 / 2,452 = 0.7488. Our period-weighted comparison score is the mean of 0.7521, 0.7431 and 0.7516, or 0.7489.

False positives account for 19.6% of all validation purchases and 43.7% of the 918 actual negatives. Customers with purchase history account for 189 of the 215 missed returns because this group contains most of the actual returns. The missed-return rates are 189 / 1,098 (17.2%) for customers with history and 26 / 35 (74.3%) for first purchases.

### Does the last purchase gap differ from the customer's usual rhythm?

Divide the days between the current and previous purchase by the mean gap between earlier purchases. A gap of 40 days is four times a 10-day rhythm but only half an 80-day rhythm. The ratio uses only information available at the current purchase. It is undefined when there are fewer than two earlier purchases, a required value is missing or the denominator is zero. The usual training-fold imputer and missing indicator handle those cases.

We compare the same forest with and without this one column. We keep 100 trees and seed 42, test leaf minima of 5 and 10, and let each split consider either the square root of the number of encoded columns or half of those columns. The five thresholds are 0.40, 0.425, 0.45, 0.475 and 0.50. Only the threshold changes when scoring a given set of probabilities.

In [12]:
def purchase_rhythm(frame):
    gap = frame['prior_mean_gap_days']
    valid_gap = (frame['prior_purchase_count'] >= 2) & (gap > 0)
    return frame['days_since_previous_purchase'] / gap.where(valid_gap)

rhythm = purchase_rhythm(purchases)
forest_data = purchases.assign(recency_over_mean_gap=rhythm)
print('Ratio available in internal validation:',
      rhythm.loc[pooled_probs.index].notna().sum(), 'of', len(pooled_probs))

Ratio available in internal validation: 1819 of 2051


In [13]:
from itertools import product

forest_thresholds = [0.40, 0.425, 0.45, 0.475, 0.50]
trial_rows = []
for inputs_name, columns in feature_sets.items():
    for use_rhythm, leaf, split_features in product([False, True], [5, 10], ['sqrt', 0.5]):
        inputs = columns + ['recency_over_mean_gap'] if use_rhythm else columns
        candidate = clone(models['Random forest']).set_params(
            min_samples_leaf=leaf, max_features=split_features)
        periods = window_probabilities(candidate, forest_data[inputs])
        pooled = pd.concat(periods)
        for threshold in forest_thresholds:
            scores = [f1_score(y.loc[p.index], p >= threshold) for p in periods]
            metrics = error_breakdown(pooled, threshold)
            trial_rows.append({
                'Inputs': inputs_name, 'Rhythm': use_rhythm, 'Leaf': leaf,
                'Split features': str(split_features), 'Threshold': threshold,
                **dict(zip(window_names, scores)), 'Mean F1': sum(scores) / len(scores),
                'Precision': metrics['Precision'], 'Recall': metrics['Recall'],
                'FP': metrics['FP'], 'FN': metrics['FN'],
            })
forest_trials = pd.DataFrame(trial_rows)
# Same inputs, leaf size and split rule as the submitted forest.
matched = forest_trials[(forest_trials['Inputs'] == 'Transactions + clients') &
                        (forest_trials['Leaf'] == 5) &
                        (forest_trials['Split features'] == 'sqrt')]
display(matched[['Rhythm', 'Threshold', 'Mean F1', 'Precision', 'Recall', 'FP', 'FN']].round(4))

,Rhythm,Threshold,Mean F1,Precision,Recall,FP,FN
40,False,0.400,0.7429,0.6592,0.8517,499,168
41,False,0.425,0.7478,0.6789,0.8323,446,190
42,False,0.450,0.7489,0.6960,0.8102,401,215
43,False,0.475,0.7503,0.7121,0.7926,363,235
44,False,0.500,0.7442,0.7270,0.7617,324,270
60,True,0.400,0.7444,0.6612,0.8526,495,167
61,True,0.425,0.7473,0.6802,0.8297,442,193
62,True,0.450,0.7447,0.6966,0.8005,395,226
63,True,0.475,0.7444,0.7127,0.7793,356,250
64,True,0.500,0.7382,0.7261,0.7511,321,282


Each precision, recall and error count pools the same validation purchases. Mean F1 gives each period equal weight. For the submitted forest, moving from threshold 0.45 to 0.475 prevents 38 false positives and misses 20 additional returns. Precision rises to 71.2%, recall falls to 79.3% and mean F1 rises to 0.7503. Lowering the threshold to 0.40 raises recall to 85.2% but adds 98 false positives and lowers mean F1 to 0.7429.

The rhythm ratio is defined for 1,819 of the 2,051 validation purchases. Adding it to the submitted forest at 0.45 lowers mean F1 from 0.7489 to 0.7447. The next comparison checks whether the effect changes with the forest settings.

In [14]:
# Select on unrounded mean F1. Exact ties prefer fewer inputs, then a higher threshold.
forest_ranked = forest_trials.sort_values(
    ['Mean F1', 'Inputs', 'Rhythm', 'Threshold'], ascending=[False, True, True, False],
    kind='stable')
leaders = forest_ranked.drop_duplicates(['Inputs', 'Rhythm'])
display(leaders.reset_index(drop=True).round(4))

# At the winning settings, change only the rhythm column.
winner = forest_ranked.iloc[0]
same_settings = forest_trials[
    (forest_trials['Inputs'] == winner['Inputs']) &
    (forest_trials['Leaf'] == winner['Leaf']) &
    (forest_trials['Split features'] == winner['Split features']) &
    (forest_trials['Threshold'] == winner['Threshold'])]
display(same_settings[['Rhythm', *window_names, 'Mean F1', 'Precision', 'Recall', 'FP', 'FN']].round(4))

,Inputs,Rhythm,Leaf,Split features,Threshold,Period 1,Period 2,Period 3,Mean F1,Precision,Recall,FP,FN
0,Transactions,True,10,0.5,0.450,0.7497,0.7544,0.7588,0.7543,0.7080,0.8067,377,219
1,Transactions + clients,True,10,0.5,0.450,0.7542,0.7520,0.7517,0.7526,0.7041,0.8085,385,217
2,Transactions + clients,False,5,sqrt,0.475,0.7525,0.7455,0.7530,0.7503,0.7121,0.7926,363,235
3,Transactions,False,10,sqrt,0.475,0.7481,0.7518,0.7470,0.7490,0.7108,0.7917,365,236


,Rhythm,Period 1,Period 2,Period 3,Mean F1,Precision,Recall,FP,FN
17,False,0.7455,0.7544,0.7436,0.7478,0.700,0.8032,390,223
37,True,0.7497,0.7544,0.7588,0.7543,0.708,0.8067,377,219


**Best internal candidate.** The forest using transactions and the rhythm ratio, 10 purchases per leaf, half the encoded columns considered per split and threshold 0.45 scored 0.7497, 0.7544 and 0.7588 (mean 0.7543). Across the same purchases it made 377 false positives and missed 219 returns. Relative to the submitted forest, this is 24 fewer false positives and four more missed returns.

**What did the ratio contribute?** At exactly those winning settings, leaving the ratio out gives mean F1 of 0.7478. Adding it gives 0.7543, with 13 fewer false positives and four fewer missed returns. F1 improved in periods 1 and 3 and tied in period 2. This measures the effect of adding the ratio and its fold-fitted missing indicator to this configuration. These related forests were selected on the same three periods, with one random seed. Section 4.7 reports the public Kaggle scores for the submitted configurations.

<a id="new-relations"></a>

### 4.6.1 Testing four more relationships

Every comparison starts with the original transaction inputs and the rhythm ratio. Each feature is calculated from the current purchase and its available history.

| Candidate | Calculation and interpretation |
|---|---|
| Relative basket value | Amount due now divided by the customer's mean past purchase value. A value of 2 means twice the usual amount. |
| Change in purchase frequency | Purchases in the latest 90 days minus purchases in the preceding 90 days. Four purchases versus one gives +3. We require at least 180 days of observed tenure. |
| Variation in purchase intervals | Standard deviation of prior gaps divided by their mean. Customers with the same average gap may differ in how regularly they buy. We need at least two prior gaps. |
| Discount share | Total line and global discounts divided by the gross purchase amount. A value of 0.20 means a 20% discount. This feature can be calculated on a first purchase. |

Undefined ratios stay missing. For the basket ratio, we checked the units in the earliest training window. In 1,893 consecutive pairs with the prior count increasing by exactly one, the increase in historical spend matched the earlier purchase's `Total a Pagar` in every pair. The feature itself uses only the current row.

In [15]:
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import SelectKBest, f_classif

recent = forest_data['purchase_count_90d']
preceding = forest_data['purchase_count_180d'] - recent
usual_value = forest_data['prior_mean_purchase_value']
usual_gap = forest_data['prior_mean_gap_days']
gross = forest_data['Total Bruto']
discount = forest_data['Total Desc_ Linha'] + forest_data['Total Desc_ Global']
relations = pd.DataFrame({
    'relative_basket': forest_data['Total a Pagar'] / usual_value.where(usual_value > 0),
    'frequency_change': (recent - preceding).where(forest_data['customer_tenure_days'] >= 180),
    'gap_variation': forest_data['prior_std_gap_days'] / usual_gap.where(
        (forest_data['prior_purchase_count'] >= 3) & (usual_gap > 0)),
    'discount_share': discount / gross.where(gross > 0),
})
relation_data = forest_data.join(relations)
display(relations.loc[pooled_probs.index].count().rename('Defined in 2,051 validation purchases').to_frame())

,"Defined in 2,051 validation purchases"
relative_basket,1917
frequency_change,1829
gap_variation,1735
discount_share,2051


We use two class methods to assess these features. First, we add each one separately to the fixed forest and compare F1 on exactly the same purchases. Second, we adapt the Week 04 filter method. In each training period, `SelectKBest` ranks the four numeric candidates by their individual association with the binary target. The ANOVA F score gives the same ranking as the absolute point-biserial correlation for these numeric inputs. We fix k at two as a simple halfway point between none and all four. A feature useful only in combination with another may receive a low rank.

For each of logistic regression, gradient boosting and random forest, we compare the rhythm baseline, all four added, and the two selected in that period. We repeat this with the four customer attributes. The 0.45 threshold and the model settings stay fixed within each comparison. All imputation and selection use only the corresponding training rows.

In [16]:
chosen_relations = []
for train_rows, _ in folds:
    train_values = SimpleImputer(strategy='median').fit_transform(relations.loc[train_rows])
    selector = SelectKBest(f_classif, k=2).fit(train_values, y.loc[train_rows])
    chosen_relations.append(relations.columns[selector.get_support()].tolist())
display(pd.DataFrame(chosen_relations, index=window_names, columns=['Selected feature A', 'Selected feature B']))

,Selected feature A,Selected feature B
Period 1,gap_variation,discount_share
Period 2,gap_variation,discount_share
Period 3,gap_variation,discount_share


In [17]:
relation_models = {
    'Random forest': clone(models['Random forest']).set_params(min_samples_leaf=10, max_features=0.5),
    'Logistic regression': models['Logistic regression'],
    'Gradient boosting': models['Gradient boosting'],
}
packages = {
    'Rhythm baseline': [[] for _ in folds],
    'All four': [relations.columns.tolist() for _ in folds],
    'Selected two': chosen_relations,
}
rows = []
for model_name, classifier in relation_models.items():
    trials = packages.copy()
    if model_name == 'Random forest':
        for column in relations:
            trials[column] = [[column] for _ in folds]
    for inputs_name, inputs in feature_sets.items():
        base = inputs + ['recency_over_mean_gap']
        for trial, extras in trials.items():
            columns_by_period = [base + extra for extra in extras]
            probabilities = window_probabilities(classifier, relation_data, columns_by_period)
            scores = [f1_score(y.loc[p.index], p >= 0.45) for p in probabilities]
            counts = error_breakdown(pd.concat(probabilities), 0.45)
            rows.append([model_name, inputs_name, trial, *scores, sum(scores) / len(scores),
                         counts['FP'], counts['FN']])

relation_results = pd.DataFrame(rows, columns=['Model', 'Inputs', 'Trial', *window_names,
                                               'Mean F1', 'FP', 'FN'])
summary = relation_results.pivot(index=['Model', 'Inputs'], columns='Trial', values='Mean F1')
display(summary[list(packages)].round(4))
forest_summary = relation_results[relation_results['Model'] == 'Random forest']
display(forest_summary[['Inputs', 'Trial', *window_names, 'Mean F1', 'FP', 'FN']].round(4).reset_index(drop=True))

Trial                                       Rhythm baseline  All four  \
Model               Inputs                                              
Gradient boosting   Transactions                     0.7344    0.7382   
                    Transactions + clients           0.7382    0.7363   
Logistic regression Transactions                     0.7326    0.7298   
                    Transactions + clients           0.7245    0.7222   
Random forest       Transactions                     0.7543    0.7470   
                    Transactions + clients           0.7526    0.7432   

Trial                                       Selected two  
Model               Inputs                                
Gradient boosting   Transactions                  0.7400  
                    Transactions + clients        0.7402  
Logistic regression Transactions                  0.7302  
                    Transactions + clients        0.7257  
Random forest       Transactions                  0.7457  
                    Transactions + clients        0.7470

,Inputs,Trial,Period 1,Period 2,Period 3,Mean F1,FP,FN
0,Transactions,Rhythm baseline,0.7497,0.7544,0.7588,0.7543,377,219
1,Transactions,All four,0.7539,0.7503,0.7368,0.7470,387,226
2,Transactions,Selected two,0.7560,0.7480,0.7331,0.7457,398,222
3,Transactions,relative_basket,0.7524,0.7486,0.7470,0.7493,399,215
4,Transactions,frequency_change,0.7524,0.7477,0.7520,0.7507,395,215
5,Transactions,gap_variation,0.7397,0.7436,0.7433,0.7422,394,232
6,Transactions,discount_share,0.7587,0.7465,0.7554,0.7535,386,215
7,Transactions + clients,Rhythm baseline,0.7542,0.7520,0.7517,0.7526,385,217
8,Transactions + clients,All four,0.7485,0.7474,0.7338,0.7432,393,230
9,Transactions + clients,Selected two,0.7527,0.7459,0.7423,0.7470,388,226


**Keep the rhythm baseline for the forest.** On transactions it scores 0.7543. Adding the four features gives 0.7470, and adding the two selected features gives 0.7457. None of the four single additions improves the mean. The closest, discount share (0.7535), catches four more returns but produces nine more false positives across the 2,051 purchases. With the same forest and rhythm settings, adding the four client attributes gives 0.7526. Clients help in the first period and reduce F1 in the next two.

**The filter selected gap variation and discount share in all three training periods.** It measures association with the target one feature at a time. A selected feature can repeat information already available to the model. For gradient boosting on transactions, the selected pair raises mean F1 from 0.7344 to 0.7400 and reduces both false positives (313 to 308) and missed returns (295 to 287). With clients, the pair raises boosting F1 from 0.7382 to 0.7402. Logistic regression shows small, mixed changes.

These comparisons hold the threshold and model settings fixed. They support keeping the simpler forest at this stage. Improvements for another model, or for one period, do not establish a gain on future purchases.

<a id="groups"></a>

### 4.6.2 Which input groups improve the forest?

We use purchases in the last 30, 90, 180 and 365 days and the lifetime purchase count as the starting point. We add timing, historical spending, current purchase details or client attributes separately. We then remove each group in turn from the full forest. A group may add little on its own yet help when combined with other inputs. Removing counts still leaves related information such as spending and tenure.

Two further comparisons change one input of the full forest. Document series may track calendar time. The year since registration can instead be expressed as the purchase year minus the registration year, so customers who joined in different years can have the same registration age. It measures whole calendar years and differs from the days of observed purchasing history.

We keep 100 trees, at least five training purchases per leaf, square-root feature sampling and a threshold of 0.45. For every input set we score the same three development periods with seeds 0, 1, 2 and 42. We rank by the mean F1 over those periods and seeds. Exact ties favour fewer inputs. We carry the highest-ranked alternative and the original forest to the later historical comparison.

In [18]:
frequency = ['prior_purchase_count'] + [f'purchase_count_{days}d' for days in [30, 90, 180, 365]]
timing = ['days_since_previous_purchase', 'customer_tenure_days', 'prior_mean_gap_days', 'prior_std_gap_days']
spending = [col for col in transaction_features if col.startswith('spend_') or 'purchase_value' in col
            or col == 'prior_total_spend']
current = [col for col in transaction_features if col not in frequency + timing + spending]
groups = {'Frequency': frequency, 'Timing': timing, 'Past spending': spending,
          'Current purchase': current, 'Clients': client_features}
assert sorted(sum(groups.values(), [])) == sorted(client_inputs)

full = 'Full forest'
variants = {full: client_inputs, 'Frequency only': frequency}
for name, group in groups.items():
    if name != 'Frequency':
        variants['Frequency + ' + name] = [col for col in client_inputs if col in frequency + group]
    variants['Without ' + name] = [col for col in client_inputs if col not in group]
variants['Without document series'] = [col for col in client_inputs if col != 'document_series']
variants['Registration age'] = [col if col != 'customer_since_year' else 'registration_age'
                                for col in client_inputs]
age = date.dt.year - pd.to_numeric(purchases['customer_since_year'])
assert age.dropna().ge(0).all()
group_data = purchases.assign(registration_age=age)

In [19]:
seeds = [0, 1, 2, 42]
group_rows = []
for name, columns in variants.items():
    for seed in seeds:
        forest = clone(reference_forest).set_params(random_state=seed)
        probs = window_probabilities(forest, group_data[columns])
        scores = [f1_score(y.loc[p.index], p >= 0.45) for p in probs]
        group_rows.append([name, len(columns), seed, *scores, sum(scores) / len(scores)])

group_trials = pd.DataFrame(group_rows, columns=['Inputs', 'Columns', 'Seed', *window_names, 'Mean F1'])
averages = group_trials.groupby('Inputs', sort=False)[['Columns', *window_names, 'Mean F1']].mean()
averages['Columns'] = averages['Columns'].astype(int)
averages['Seed SD'] = group_trials.groupby('Inputs')['Mean F1'].std()
averages['Change vs full'] = averages['Mean F1'] - averages.loc[full, 'Mean F1']
ranking = averages.sort_values(['Mean F1', 'Columns'], ascending=[False, True], kind='stable')
challenger = ranking.drop(index=full).index[0]
display(ranking.round(4))

,Columns,Period 1,Period 2,Period 3,Mean F1,Seed SD,Change vs full
Inputs,,,,,,,
Without Frequency,33,0.7491,0.7482,0.7505,0.7492,0.0034,0.0008
Without Timing,34,0.7497,0.7526,0.7435,0.7486,0.0029,0.0001
Full forest,38,0.7508,0.7478,0.7468,0.7485,0.0038,0.0000
Registration age,38,0.7477,0.7489,0.7459,0.7475,0.0029,-0.0010
Without Clients,34,0.7461,0.7507,0.7421,0.7463,0.0033,-0.0022
Without Past spending,29,0.7428,0.7591,0.7337,0.7452,0.0021,-0.0033
Frequency + Current purchase,21,0.7434,0.7586,0.7320,0.7446,0.0026,-0.0038
Without document series,37,0.7467,0.7490,0.7365,0.7441,0.0043,-0.0044
Frequency + Past spending,14,0.7406,0.7456,0.7445,0.7436,0.0011,-0.0049


The full forest averaged **0.7485** over the three development periods and four seeds. Removing all five count inputs ranked first at **0.7492**, a difference of **0.0008** using unrounded scores. The forest still sees spending in rolling windows, time since the last purchase and mean purchase gaps, which also describe buying activity. The forest given only the five counts scored 0.7321. With this configuration, current purchase details and historical spending each added more to the count-only model than client attributes alone.

Removing the four customer attributes from the full forest lowered mean F1 to 0.7463. Dropping document series gave 0.7441, and replacing registration year with registration age gave 0.7475. We carry the forest without explicit counts forward as the development challenger.

<a id="segments"></a>

## IV. Open-ended section: When does the forest improve on purchase frequency?

A team planning customer follow-up could use a rule it can apply by hand: predict a return when the customer made at least three purchases in the previous 365 days. We fix this cutoff for the comparison and also include predicting a return for everyone. We use known outcomes in labelled data to ask where the forest avoids unnecessary follow-ups and where it misses actual returns.

First we compare the two forests on the three earlier periods and the 1,942 purchases from 13 July 2022 to 15 February 2025. Training for that last block stops before the 90-day gap. This block already informed earlier project comparisons. It serves as a historical cross-check. Each forest score below averages four seeds. The two simple rules give the same answer on every run.

In [20]:
recent_train = date < development_end
recent_valid = date >= main_start
assert date[recent_train].max() + horizon < date[recent_valid].min()
recent_rows = [(recent_train, recent_valid)]
methods = {}
for name, prediction in [('Always positive', pd.Series(1, index=y.index)),
                         ('At least 3 purchases', purchases['purchase_count_365d'] >= 3)]:
    methods[name] = [f1_score(y.loc[v], prediction.loc[v]) for _, v in folds + recent_rows]

for name in [full, challenger]:
    recent_scores = []
    for seed in seeds:
        forest = clone(reference_forest).set_params(random_state=seed)
        p, = window_probabilities(forest, group_data[variants[name]], windows=recent_rows)
        recent_scores.append(f1_score(y.loc[p.index], p >= 0.45))
        if name == full and seed == 42:
            recent_forest = p >= 0.45
    methods[name] = [*averages.loc[name, window_names], sum(recent_scores) / len(recent_scores)]

comparison = pd.DataFrame.from_dict(methods, orient='index', columns=[*window_names, 'Recent block'])
comparison['Earlier mean'] = comparison[window_names].mean(axis=1)
display(comparison[[*window_names, 'Earlier mean', 'Recent block']].round(4))

,Period 1,Period 2,Period 3,Earlier mean,Recent block
Always positive,0.7063,0.7464,0.6788,0.7105,0.7211
At least 3 purchases,0.7233,0.7526,0.7261,0.7340,0.7545
Full forest,0.7508,0.7478,0.7468,0.7485,0.7607
Without Frequency,0.7491,0.7482,0.7505,0.7492,0.7603


For the error analysis we use seed 42, the configuration actually submitted. We group purchases by the customer's purchase count **before** the current occasion. The last two columns subtract the frequency rule's error counts from the forest's. A negative number means the forest made fewer errors of that kind.

In [21]:
counts = purchases.loc[recent_valid, 'purchase_count_365d']
assert counts.notna().all() and counts.ge(0).all()
bands = pd.cut(counts, [-1, 0, 2, 5, 11, float('inf')], labels=['0', '1 to 2', '3 to 5', '6 to 11', '12+'])
segment_data = pd.DataFrame({'Band': bands, 'Actual': y.loc[recent_valid],
                             'Rule': counts >= 3, 'Forest': recent_forest})
print('Recent F1 for the submitted forest (seed 42):', round(f1_score(segment_data['Actual'], recent_forest), 4))
segment_rows = []
for band, group in segment_data.groupby('Band', observed=True):
    actual = group['Actual']
    rule = group['Rule']
    forest = group['Forest']
    fp_change = (forest & actual.eq(0)).sum() - (rule & actual.eq(0)).sum()
    fn_change = (~forest & actual.eq(1)).sum() - (~rule & actual.eq(1)).sum()
    segment_rows.append([band, len(group), 100 * actual.mean(), f1_score(actual, [1] * len(group)),
                         f1_score(actual, rule), f1_score(actual, forest), fp_change, fn_change])

segments = pd.DataFrame(segment_rows, columns=['Prior year purchases', 'Purchases', 'Return %',
                         'F1 all positive', 'F1 rule', 'F1 forest', 'Change in FP', 'Change in FN'])
assert segments['Purchases'].sum() == recent_valid.sum()
display(segments.set_index('Prior year purchases').round(3))

Recent F1 for the submitted forest (seed 42): 0.7587


,Purchases,Return %,F1 all positive,F1 rule,F1 forest,Change in FP,Change in FN
Prior year purchases,,,,,,,
0,180,25.556,0.407,0.000,0.344,7,-11
1 to 2,490,31.837,0.483,0.000,0.361,89,-54
3 to 5,658,54.255,0.703,0.703,0.698,-117,67
6 to 11,409,81.907,0.901,0.901,0.896,0,3
12+,205,98.049,0.990,0.990,0.990,0,0


**Model choice.** The earlier advantage from removing the counts did not carry into the recent block. Across the four seeds the full forest scored **0.7607**, the forest without counts **0.7603** and the frequency rule **0.7545**. We retain the full forest and its 38 inputs for deployment. The forest without counts also scored 0.80000 public F1 on Kaggle (submission `58691641`).

**Where decisions change.** Among the 658 occasions with three to five prior-year purchases, using the submitted forest avoided 117 false positives and missed 67 extra returns compared with the rule. F1 in that group fell slightly, from 0.703 to 0.698. With one or two purchases, the rule predicts no returns. The forest caught 54 additional returns at the cost of 89 extra false positives. With twelve or more purchases, the methods made identical predictions on all 205 occasions. Which trade-off is preferable depends on the cost of following up and of missing a returning customer.

The frequency rule reproduces all 986 predictions in the official sample submission, as checked by ID in Section V. On the public leaderboard the sample scored 0.77941 and the selected forest scored 0.80798. Individual test outcomes are unavailable, so the segment breakdown uses labelled historical purchases.

<a id="threshold-history"></a>

### Do different thresholds or shorter training histories improve global F1?

We make two changes separately to the 38-input client forest. For the threshold experiment we keep 0.45 outside the group with one to five purchases in the previous 365 days and try 0.40 or 0.50 inside it. A comparison using each threshold for everyone tells us whether the group-specific decision adds anything. For the training experiment we keep 0.45 for everyone and retain either two or four years of eligible training purchases counted back from the end of the training window. Historical predictors on each retained purchase keep their original lookback. The reference uses all eligible history and 0.45 throughout.

We choose one alternative per family using mean F1 on **all purchases** in the three earlier periods, averaged across seeds 0, 1, 2 and 42. Ties follow table order. We then freeze those choices and refit at the start of each of three consecutive later periods. Every fit ends before the 90-day outcome gap. A later refit may use a past validation purchase once its outcome is available. This evaluates periodic retraining, whereas our earlier 1,942-purchase check used a single fit. We previously examined errors in that block, so these are development checks.

**Kaggle decision fixed before scoring.** A candidate must beat the reference on the earlier and later mean F1, win in at least two of the three later periods and improve the later mean for submission seed 42. If several qualify, we submit the one with the highest earlier mean, once.

In [22]:
later_starts = [main_start + pd.DateOffset(years=i) for i in range(3)]
later_edges = later_starts + [date.max() + pd.Timedelta(days=1)]
later_folds = [(date < start - horizon, (date >= start) & (date < end))
               for start, end in zip(later_starts, later_edges[1:])]
assert sum(valid.astype(int) for _, valid in later_folds).equals(recent_valid.astype(int))
assert max(date[valid].max() for _, valid in folds) + horizon < main_start

# Years of eligible training data, threshold for 1 to 5 purchases, threshold elsewhere.
policies = {
    'Original': (None, 0.45, 0.45),
    'Global 0.40': (None, 0.40, 0.40), 'Global 0.50': (None, 0.50, 0.50),
    'Group 0.40': (None, 0.40, 0.45), 'Group 0.50': (None, 0.50, 0.45),
    'Four years': (4, 0.45, 0.45), 'Two years': (2, 0.45, 0.45),
}
in_group = purchases['purchase_count_365d'].between(1, 5)

def training_window(train, cutoff, years):
    if years is None:
        return train
    return train & date.ge(cutoff - pd.DateOffset(years=years))

window_sizes = []
for start, (train, valid) in zip(later_starts, later_folds):
    cutoff = start - horizon
    window_sizes.append([start.date(), date[valid].max().date(), int(valid.sum()),
                         int((valid & in_group).sum()), int(train.sum()),
                         int(training_window(train, cutoff, 4).sum()),
                         int(training_window(train, cutoff, 2).sum())])
display(pd.DataFrame(window_sizes, columns=['From', 'Last purchase', 'Validation', '1 to 5',
                                            'Train all', 'Train 4 years', 'Train 2 years']))

,From,Last purchase,Validation,1 to 5,Train all,Train 4 years,Train 2 years
0,2022-07-13,2023-07-11,725,449,4424,2774,1432
1,2023-07-13,2024-07-07,741,435,5101,2780,1415
2,2024-07-13,2025-02-15,476,264,5883,2891,1459


In [23]:
def score_policies(windows, starts, names):
    rows = []
    for period, ((train, valid), start) in enumerate(zip(windows, starts, strict=True), start=1):
        cutoff = start - horizon
        assert date[train].max() + horizon < start <= date[valid].min()
        for years in dict.fromkeys(policies[name][0] for name in names):
            fit_rows = training_window(train, cutoff, years)
            for seed in seeds:
                forest = clone(reference_forest).set_params(random_state=seed)
                p, = window_probabilities(forest, purchases[client_inputs], windows=[(fit_rows, valid)])
                for name in names:
                    duration, inside, outside = policies[name]
                    if duration != years:
                        continue
                    threshold = pd.Series(outside, index=p.index).where(~in_group.loc[p.index], inside)
                    actual = y.loc[p.index]
                    predicted = p >= threshold
                    rows.append([name, period, seed, f1_score(actual, predicted),
                                 (predicted & actual.eq(0)).sum(), (~predicted & actual.eq(1)).sum()])
    return pd.DataFrame(rows, columns=['Policy', 'Period', 'Seed', 'F1', 'FP', 'FN'])

def period_table(scores):
    table = scores.pivot_table(index='Policy', columns='Period', values='F1', sort=False)
    table.columns = [f'Period {i}' for i in table.columns]
    table['Mean'] = table.mean(axis=1)
    return table

early_scores = score_policies(folds, boundaries[:3], list(policies))
early = period_table(early_scores)
families = [['Global 0.40', 'Global 0.50'], ['Group 0.40', 'Group 0.50'], ['Four years', 'Two years']]
frozen = ['Original'] + [early.loc[family, 'Mean'].idxmax() for family in families]
display(early.loc[list(policies)].round(4))
print('Fixed before the later comparison:', ', '.join(frozen))

,Period 1,Period 2,Period 3,Mean
Policy,,,,
Original,0.7508,0.7478,0.7468,0.7485
Global 0.40,0.7515,0.7493,0.7398,0.7468
Global 0.50,0.7420,0.7420,0.7409,0.7416
Group 0.40,0.7546,0.7479,0.7422,0.7482
Group 0.50,0.7409,0.7442,0.7393,0.7414
Four years,0.7508,0.7502,0.7357,0.7456
Two years,0.7406,0.7451,0.7273,0.7377


Fixed before the later comparison: Original, Global 0.40, Group 0.40, Four years


In [24]:
late_scores = score_policies(later_folds, later_starts, frozen)
late = period_table(late_scores).loc[frozen]
delta = late.subtract(late.loc['Original'], axis='columns')
seed42 = late_scores[late_scores['Seed'] == 42].groupby('Policy')
seed42_mean = seed42['F1'].mean()
decision = late.copy()
decision['Earlier gain'] = early['Mean'] - early.loc['Original', 'Mean']
decision['Later gain'] = delta['Mean']
decision['Later wins'] = delta.iloc[:, :3].gt(0).sum(axis=1)
decision['Seed 42 gain'] = seed42_mean - seed42_mean['Original']
for error in ['FP', 'FN']:
    totals = seed42[error].sum()
    decision['Change in ' + error] = totals - totals['Original']
qualifies = ((decision['Earlier gain'] > 0) & (decision['Later gain'] > 0)
             & (decision['Later wins'] >= 2) & (decision['Seed 42 gain'] > 0))
eligible = decision.index[qualifies]
kaggle_candidate = early.loc[eligible, 'Mean'].idxmax() if len(eligible) else None
display(late.round(4))
display(decision.iloc[:, 4:].round(4))
print('Candidate for Kaggle:', kaggle_candidate)

,Period 1,Period 2,Period 3,Mean
Policy,,,,
Original,0.7123,0.7681,0.8065,0.7623
Global 0.40,0.7100,0.7808,0.8035,0.7648
Group 0.40,0.7131,0.7804,0.8019,0.7651
Four years,0.7066,0.7583,0.8042,0.7564


,Earlier gain,Later gain,Later wins,Seed 42 gain,Change in FP,Change in FN
Policy,,,,,,
Original,0.0000,0.0000,0,0.0000,0,0
Global 0.40,-0.0016,0.0025,1,-0.0012,95,-59
Group 0.40,-0.0003,0.0028,2,-0.0002,82,-53
Four years,-0.0029,-0.0059,0,-0.0123,-22,36


Candidate for Kaggle: None


**Threshold.** Lowering the cut to 0.40 for the target group moved earlier mean F1 from 0.7485 to 0.7482. With annual refits it improved the later mean from 0.7623 to 0.7651 across four seeds, winning the first two periods and losing the third. Lowering the cut for everyone scored 0.7648 on the same later periods. For the actual submission seed, the group-specific cut raised false positives from 353 to 435 and reduced missed returns from 202 to 149. Its equally weighted later F1 was 0.7662, compared with 0.7664 for the original. Error counts pool individual purchases, while mean F1 gives each period equal weight.

**Training history.** In the earlier periods, mean F1 was 0.7377 with two years, 0.7456 with four years and 0.7485 with all eligible history. Four years was the stronger restricted window. It scored 0.7564 on the later periods and lost to all history in each of them. Restricting history also reduces the number of training purchases, so this comparison measures both changes together.

**Decision.** No change met the criteria for a Kaggle submission. We keep the full client forest trained on all eligible purchases with a single threshold of 0.45. Its previously confirmed public F1 is 0.80798.

### Scope of the results

Each purchase gets one vote in its period, so customers with many purchases contribute more observations. The same customer can appear on both sides of a cutoff. Performance for entirely new customers is a separate question.

The three internal periods were used to compare models, input sets, settings and thresholds. The best internal mean is therefore a selection score. Variation across the three periods shows how a result changed with time. The periods share much of their training data, so small differences cannot establish statistical significance. Giving periods equal weight also gives the shorter third period the same influence as each earlier year.

Earlier EDA examined target relationships and Notebook 03 used the later 1,942 purchases to compare logistic models. The three earlier periods supply the development scores. Section IV uses the later block for a fixed-fit historical check and then evaluates annual refits on the same purchases. The segment analysis helped motivate the threshold experiment, so the later scores also reflect prior exploration. Its purchases also enter the refit for the Kaggle submission, whose public score comes from different purchase occasions.

Course methods come from Week 04 (preprocessing and feature selection), Week 05 (assessment and metrics), Weeks 06 to 10 (the first five classifiers), Week 11 (boosting), Week 12 (the MLP) and Week 13 (regularisation). The three calendar windows, 90-day gap and same-row timing identity follow from the purchase task and the supplied data.

<a id="kaggle-rf"></a>

## V. Deployment

### 4.7 Reproducing the Kaggle submissions

The exports below refit the preprocessing and forest on all 6,469 eligible purchases. Their 90-day outcomes were known before the first test purchase. We apply the same row-level corrections to test data and match the predictions to the 986 IDs in the sample file. Each configuration writes a separate CSV.

The first submission on 20 November 2026 used four client attributes, 100 trees, five purchases per leaf and the default square-root feature sampling. At threshold 0.45 its public F1 was **0.80798** (submission `58617907`).

The first cell below is also the standalone deployment entry point. It reads the official CSVs, fits the selected forest and writes the file used for our best public submission. From the repository root run `python -m src.final_forest` to reproduce just that file.

In [25]:
output_dir = root / 'outputs' / 'notebook04'
client_submission = write_selected_submission(root / 'data', output_dir)
print(SELECTED_FILE, '| Predictions:', len(client_submission), '| Positive:', client_submission[target].sum())

submission_rf_clients_leaf5_t045.csv | Predictions: 986 | Positive: 726


### Transactions with purchase rhythm

The candidate from section 4.6 uses transactions and the single purchase rhythm ratio. Use the same 100 trees, seed 42, 10 purchases per leaf, half of the encoded columns considered per split and threshold 0.45. Its mean F1 across the three internal periods was 0.7543. Section 4.6.1 tested four further variables and found that this forest was strongest with the rhythm ratio alone.

In [26]:
test = pd.read_csv(root / 'data' / 'test.csv')
sample = pd.read_csv(root / 'data' / 'sample_submission.csv')
sample_rule = test.set_index('ID')['purchase_count_365d'].ge(3).astype(int)
assert sample['ID'].map(sample_rule).equals(sample[target])
future, _ = repair_history(test)
future[time_columns] = recover_time_intervals(future)

rhythm_columns = transaction_features + ['recency_over_mean_gap']
rhythm_forest = clone(models['Random forest']).set_params(min_samples_leaf=10, max_features=0.5)
future_rhythm = future.assign(recency_over_mean_gap=purchase_rhythm(future))
rhythm_submission = export_forest(rhythm_forest, forest_data, future_rhythm, rhythm_columns, 0.45,
                                 sample, output_dir / 'submission_rf_transactions_rhythm_leaf10_t045.csv')
print('Predictions differing from the client forest:', rhythm_submission[target].ne(client_submission[target]).sum())

Predictions differing from the client forest: 61


**Public Kaggle result.** On 21 November 2026 this forest scored **0.79487** (submission `58677446`). It changed 61 of the 986 predictions. The earlier forest with customer attributes scored **0.80798**, so the ranking of these two forests on Kaggle was the reverse of their ranking across the three internal periods. The submissions also differ in leaf size and the number of features considered at each split. The public comparison cannot isolate the effect of the rhythm ratio or the customer attributes.

### Adding purchase rhythm to the original client forest

We keep the four customer attributes, the original forest settings and the 0.45 threshold. The one added input is the time since the previous purchase divided by the customer's historical mean gap. Missing ratios go through the same numerical preprocessing as the other inputs.

In [27]:
client_rhythm_columns = feature_sets['Transactions + clients'] + ['recency_over_mean_gap']
client_rhythm_submission = export_forest(
    reference_forest, forest_data, add_client_features(future_rhythm, clients), client_rhythm_columns,
    0.45, sample, output_dir / 'submission_rf_clients_rhythm_leaf5_t045.csv'
)
print('Predictions differing from the original client forest:',
      client_rhythm_submission[target].ne(client_submission[target]).sum())

Predictions differing from the original client forest: 47


On 21 November 2026, adding the rhythm ratio changed 47 of the 986 predictions. Public F1 went from **0.80798** to **0.80299** (submission `58677954`). For this forest and threshold, the original inputs gave the higher public F1. Both CSVs were generated using scikit-learn 1.9.0. The refitted original matched all 986 predictions in the earlier Kaggle file.

### Client forest without purchase counts

Removing the five frequency inputs kept 33 inputs and changed 47 predictions. This variant scored 0.80000 on 21 November 2026.

In [28]:
no_counts_submission = export_forest(
    reference_forest, purchases, add_client_features(future, clients), variants['Without Frequency'],
    0.45, sample, output_dir / 'submission_rf_clients_without_counts_leaf5_t045.csv')
print('Predictions differing from the original:', no_counts_submission[target].ne(client_submission[target]).sum())

Predictions differing from the original: 47


**Selected output.** Use `outputs/notebook04/submission_rf_clients_leaf5_t045.csv` for the full client forest (public F1 0.80798). The standalone command `python -m src.final_forest` rebuilds this file from the supplied data.